# Diagnostics notebook

Original environment, ChEMBL dataset/vocabulary, graph-compatibility, BindingDB leakage and train/validation split audit cells, unchanged. Includes the original split creation cells so it runs from a clean kernel; does not run the neural-network pretraining loop.

# **PRETRAINING WITH NO TOXICITY OBJECTIVE**

## Environment and input-file verification

In [13]:
from pathlib import Path
import os
import sys
import importlib
from itertools import islice

In [14]:
PROJECT_ROOT = Path(
    os.environ.get("CTDDG_REPO_ROOT", "/home/muhamed/repo/CTDDG")
).expanduser().resolve()

PREPROCESSING_ROOT = PROJECT_ROOT / "REWORKED_CODE" / "PREPROCESSING_OUTPUT"

CHEMBL_DIR = PREPROCESSING_ROOT / "chembl"
CLEAN_TXT = CHEMBL_DIR / "chembl_clean.txt"
CLEAN_CSV = CHEMBL_DIR / "chembl_clean.csv"

ATOM_TYPES_PATH = (
    PREPROCESSING_ROOT / "vocabulary" / "atom_types.txt"
)

# Do not choose the dataset until its format is verified.
CHEMBL_PATH = None

In [15]:
print("=" * 65)
print("SECTION 1A — ENVIRONMENT")
print("=" * 65)

print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version.split()[0]}")
print(f"Working directory : {Path.cwd()}")
print(f"Project root      : {PROJECT_ROOT}")

for package in ["torch", "rdkit", "numpy", "scipy", "networkx"]:
    try:
        module = importlib.import_module(package)
        print(f"{package:12s} : {getattr(module, '__version__', 'installed')}")
    except Exception as exc:
        print(f"{package:12s} : ERROR — {exc}")

try:
    import torch
    print(f"CUDA available    : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"GPU {i}             : {props.name}")
            print(f"GPU memory        : {props.total_memory / 2**30:.2f} GiB")
except Exception as exc:
    print(f"GPU inspection    : ERROR — {exc}")

SECTION 1A — ENVIRONMENT
Python executable : /home/muhamed/.conda/envs/ctddg_torch/bin/python
Python version    : 3.11.16
Working directory : /home/muhamed/repo/CTDDG/REWORKED_CODE/testing
Project root      : /home/muhamed/repo/CTDDG
torch        : 2.4.1+cu121
rdkit        : 2026.03.6
numpy        : 2.4.6
scipy        : 1.17.1
networkx     : 3.6.1
CUDA available    : True
GPU 0             : NVIDIA L40
GPU memory        : 44.37 GiB
GPU 1             : NVIDIA L40
GPU memory        : 44.37 GiB


In [16]:
print("\n" + "=" * 65)
print("SECTION 1B — INPUT FILES")
print("=" * 65)

input_files = {
    "ChEMBL clean TXT": CLEAN_TXT,
    "ChEMBL clean CSV": CLEAN_CSV,
    "Final atom vocabulary": ATOM_TYPES_PATH,
}

for label, path in input_files.items():
    print(f"\n[{label}]")
    print(f"Path   : {path}")
    print(f"Exists : {path.is_file()}")

    if not path.is_file():
        continue

    print(f"Size   : {path.stat().st_size / 2**20:.2f} MiB")

    try:
        with path.open("r", encoding="utf-8-sig") as f:
            preview = [
                line.rstrip("\n\r")
                for line in islice(f, 5)
            ]

        print("First 5 lines:")
        for i, line in enumerate(preview, 1):
            print(f"  {i}: {line[:220]!r}")
    except Exception as exc:
        print(f"Read error: {exc}")


SECTION 1B — INPUT FILES

[ChEMBL clean TXT]
Path   : /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/chembl_clean.txt
Exists : True
Size   : 46.19 MiB
First 5 lines:
  1: 'Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1'
  2: 'O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1'
  3: 'CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4)CC3)c12'
  4: 'Cn1cccc/c1=N\\C1CC1'
  5: 'CCCCc1ccc(NC(=S)NN2CCN(C)CC2)cc1'

[ChEMBL clean CSV]
Path   : /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/chembl_clean.csv
Exists : True
Size   : 120.26 MiB
First 5 lines:
  1: 'chembl_id,original_smiles,canonical_smiles,num_atoms,num_bonds,fragmented'
  2: 'CHEMBL1902090,Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1,Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1,25,27,False'
  3: 'CHEMBL1802565,O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1,O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1,23,25,False'
  4: 'CHEMBL1531434,CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4)CC3)c12,CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4

In [17]:
print("\n" + "=" * 65)
print("SECTION 1C — VOCABULARY CHECK")
print("=" * 65)

if ATOM_TYPES_PATH.is_file():
    try:
        rows = [
            line.strip()
            for line in ATOM_TYPES_PATH.read_text(
                encoding="utf-8-sig"
            ).splitlines()
            if line.strip()
        ]

        parsed = []
        for line in rows:
            fields = [part.strip() for part in line.split(",")]
            if len(fields) != 3:
                raise ValueError(f"Invalid vocabulary row: {line!r}")
            parsed.append(
                (fields[0], int(fields[1]), int(fields[2]))
            )

        print(f"Atom type count  : {len(parsed)}")
        print(f"Unique atom types: {len(set(parsed))}")
        print(f"Duplicates       : {len(parsed) - len(set(parsed))}")
        print(f"First 5 entries  : {parsed[:5]}")
        print(f"Last 5 entries   : {parsed[-5:]}")

        assert parsed, "Vocabulary is empty"
        assert len(parsed) == len(set(parsed)), "Duplicate atom types"

        print("Vocabulary status: PASS")
    except Exception as exc:
        print(f"Vocabulary status: FAIL — {exc}")
else:
    print("Vocabulary status: MISSING")


SECTION 1C — VOCABULARY CHECK
Atom type count  : 55
Unique atom types: 55
Duplicates       : 0
First 5 entries  : [('B', -1, 0), ('B', -1, 3), ('B', 0, 0), ('Br', 0, 0), ('C', -1, 0)]
Last 5 entries   : [('Se', 0, 0), ('Si', 0, 0), ('Sn', 0, 0), ('Te', 0, 0), ('V', 0, 0)]
Vocabulary status: PASS


In [18]:
print("\n" + "=" * 65)
print("SECTION 1D — READINESS")
print("=" * 65)

dataset_available = CLEAN_TXT.is_file() or CLEAN_CSV.is_file()
vocabulary_available = ATOM_TYPES_PATH.is_file()

print(f"Clean ChEMBL available : {dataset_available}")
print(f"Vocabulary available   : {vocabulary_available}")
print("Objective              : unconditional NLL")
print("Toxicity labels        : NOT USED")
print("GPU training started   : NO")
print("Files modified         : NONE")

if dataset_available and vocabulary_available:
    print("\nSTATUS: Ready for dataset-content validation.")
else:
    print("\nSTATUS: Resolve missing inputs before continuing.")


SECTION 1D — READINESS
Clean ChEMBL available : True
Vocabulary available   : True
Objective              : unconditional NLL
Toxicity labels        : NOT USED
GPU training started   : NO
Files modified         : NONE

STATUS: Ready for dataset-content validation.


## ChEMBL Dataset and Vocabulary Inspection

Objective: Understand the actual structure of the cleaned ChEMBL datasets and the final atom vocabulary before implementing the data loader.

In [19]:
from pathlib import Path
import csv

print("=" * 65)
print("2.1 — DATASET FORMAT INSPECTION")
print("=" * 65)

for name, path in [
    ("ChEMBL TXT", CLEAN_TXT),
    ("ChEMBL CSV", CLEAN_CSV),
]:
    print(f"\n{'-' * 55}")
    print(f"FILE: {name}")
    print(f"{'-' * 55}")

    if not path.is_file():
        print("STATUS: MISSING")
        continue

    print(f"Path       : {path}")
    print(f"Size (MB)  : {path.stat().st_size / 1024**2:.2f}")

    with path.open("r", encoding="utf-8-sig", newline="") as f:
        lines = [f.readline().rstrip("\r\n") for _ in range(5)]

    print("\nFirst 5 raw records:")
    for i, line in enumerate(lines, 1):
        print(f"  Row {i}: {line[:250]!r}")

    if path.suffix.lower() == ".csv":
        try:
            parsed = list(csv.reader(lines))
            print("\nCSV column counts:", [len(row) for row in parsed])
            print("First parsed CSV row:", parsed[0][:10] if parsed else [])
        except Exception as exc:
            print("CSV parsing error:", exc)

print("\nSTATUS: Format inspection completed.")

2.1 — DATASET FORMAT INSPECTION

-------------------------------------------------------
FILE: ChEMBL TXT
-------------------------------------------------------
Path       : /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/chembl_clean.txt
Size (MB)  : 46.19

First 5 raw records:
  Row 1: 'Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1'
  Row 2: 'O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1'
  Row 3: 'CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4)CC3)c12'
  Row 4: 'Cn1cccc/c1=N\\C1CC1'
  Row 5: 'CCCCc1ccc(NC(=S)NN2CCN(C)CC2)cc1'

-------------------------------------------------------
FILE: ChEMBL CSV
-------------------------------------------------------
Path       : /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/chembl_clean.csv
Size (MB)  : 120.26

First 5 raw records:
  Row 1: 'chembl_id,original_smiles,canonical_smiles,num_atoms,num_bonds,fragmented'
  Row 2: 'CHEMBL1902090,Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1,Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1

In [20]:
from collections import deque

print("=" * 65)
print("2.2 — DATASET SIZE AND INTEGRITY")
print("=" * 65)

DATASET_STATS = {}

for name, path in [
    ("chembl_clean.txt", CLEAN_TXT),
    ("chembl_clean.csv", CLEAN_CSV),
]:
    if not path.is_file():
        continue

    total_lines = 0
    blank_lines = 0
    last_records = deque(maxlen=3)

    with path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            total_lines += 1

            if not line.strip():
                blank_lines += 1
            else:
                last_records.append(line.strip()[:150])

    DATASET_STATS[name] = {
        "total_lines": total_lines,
        "blank_lines": blank_lines,
        "nonempty_lines": total_lines - blank_lines,
    }

    print(f"\nFILE: {name}")
    print(f"Total lines       : {total_lines:,}")
    print(f"Blank lines       : {blank_lines:,}")
    print(f"Non-empty lines   : {total_lines - blank_lines:,}")
    print("Last 3 records:")
    for record in last_records:
        print(f"  {record}")

print("\nSTATUS: Dataset counting completed.")

2.2 — DATASET SIZE AND INTEGRITY

FILE: chembl_clean.txt
Total lines       : 1,090,506
Blank lines       : 0
Non-empty lines   : 1,090,506
Last 3 records:
  CC1CCCCC1(c1cccs1)N1CCCCC1
  CN1c2c(oc(=O)n(-c3ccccn3)c2=O)-c2ccccc2S1(=O)=O
  COc1cc([C@@H]2c3cc4c(cc3[C@H](O)[C@H]3COC(=O)[C@@H]23)OCO4)cc(OC)c1OC

FILE: chembl_clean.csv
Total lines       : 1,090,507
Blank lines       : 0
Non-empty lines   : 1,090,507
Last 3 records:
  CHEMBL3989791,CC1CCCCC1(c1cccs1)N1CCCCC1,CC1CCCCC1(c1cccs1)N1CCCCC1,18,20,False
  CHEMBL1213420,CN1c2c(oc(=O)n(-c3ccccn3)c2=O)-c2ccccc2S1(=O)=O,CN1c2c(oc(=O)n(-c3ccccn3)c2=O)-c2ccccc2S1(=O)=O,25,28,False
  CHEMBL283120,COc1cc([C@@H]2c3cc4c(cc3[C@H](O)[C@H]3COC(=O)[C@@H]23)OCO4)cc(OC)c1OC,COc1cc([C@@H]2c3cc4c(cc3[C@H](O)[C@H]3COC(=O)[C@@H]23)OCO4)cc(OC)c1

STATUS: Dataset counting completed.


In [21]:
from collections import Counter
import hashlib

print("=" * 65)
print("2.3 — FINAL ATOM VOCABULARY")
print("=" * 65)

with ATOM_TYPES_PATH.open("r", encoding="utf-8-sig") as f:
    raw_rows = [
        line.strip()
        for line in f
        if line.strip()
    ]

atom_types = []

for row_num, row in enumerate(raw_rows, 1):
    fields = [x.strip() for x in row.split(",")]

    if len(fields) != 3:
        raise ValueError(
            f"Invalid vocabulary entry at row {row_num}: {row!r}"
        )

    symbol = fields[0]
    formal_charge = int(fields[1])
    explicit_h = int(fields[2])

    atom_types.append((symbol, formal_charge, explicit_h))

duplicates = [
    atom for atom, count in Counter(atom_types).items()
    if count > 1
]

ATOM_VOCAB_HASH = hashlib.sha256(
    ATOM_TYPES_PATH.read_bytes()
).hexdigest()

print(f"Vocabulary path : {ATOM_TYPES_PATH}")
print(f"Total entries   : {len(atom_types)}")
print(f"Unique entries  : {len(set(atom_types))}")
print(f"Duplicates      : {len(duplicates)}")
print(f"SHA256          : {ATOM_VOCAB_HASH}")

print("\nAtom types (first 15):")
for i, atom in enumerate(atom_types[:15]):
    print(f"  {i:3d}: {atom}")

print("\nAtom types (last 5):")
for i in range(max(0, len(atom_types) - 5), len(atom_types)):
    print(f"  {i:3d}: {atom_types[i]}")

print("\nVocabulary validation:")
assert atom_types, "Empty atom vocabulary"
assert not duplicates, f"Duplicate atom types: {duplicates}"

print("STATUS: PASS")

2.3 — FINAL ATOM VOCABULARY
Vocabulary path : /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/vocabulary/atom_types.txt
Total entries   : 55
Unique entries  : 55
Duplicates      : 0
SHA256          : 3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f

Atom types (first 15):
    0: ('B', -1, 0)
    1: ('B', -1, 3)
    2: ('B', 0, 0)
    3: ('Br', 0, 0)
    4: ('C', -1, 0)
    5: ('C', -1, 1)
    6: ('C', 0, 0)
    7: ('C', 0, 1)
    8: ('C', 0, 2)
    9: ('C', 0, 3)
   10: ('C', 1, 0)
   11: ('Cl', 0, 0)
   12: ('Cl', 1, 0)
   13: ('F', 0, 0)
   14: ('Fe', -2, 0)

Atom types (last 5):
   50: ('Se', 0, 0)
   51: ('Si', 0, 0)
   52: ('Sn', 0, 0)
   53: ('Te', 0, 0)
   54: ('V', 0, 0)

Vocabulary validation:
STATUS: PASS


## Molecular Dataset Integrity and Compatibility

Verify that our cleaned dataset is suitable for graph-based pretraining before building the graph traversal and model architecture.

We'll check dataset alignment, molecular validity, compatibility with the final vocabulary, and potential graph-size problems. All checks are read-only.

In [22]:
import csv
from collections import Counter
from itertools import zip_longest

print("=" * 65)
print("3.1 — DATASET CONSISTENCY AND SIZE AUDIT")
print("=" * 65)

alignment_errors = []
metadata_errors = []
fragment_counts = Counter()

total_rows = 0
max_atoms = 0
max_bonds = 0
large_molecules = 0
missing_records = 0

with CLEAN_TXT.open("r", encoding="utf-8-sig") as txt_file, \
     CLEAN_CSV.open("r", encoding="utf-8-sig", newline="") as csv_file:

    reader = csv.DictReader(csv_file)

    print("CSV columns:", reader.fieldnames)

    required_columns = {
        "canonical_smiles",
        "num_atoms",
        "num_bonds",
        "fragmented",
    }

    missing_columns = required_columns - set(reader.fieldnames or [])
    if missing_columns:
        raise ValueError(f"Missing CSV columns: {missing_columns}")

    for row_num, (txt_line, csv_row) in enumerate(
        zip_longest(txt_file, reader),
        start=1
    ):
        total_rows += 1

        if txt_line is None or csv_row is None:
            missing_records += 1
            if len(alignment_errors) < 5:
                alignment_errors.append(
                    (row_num, "Record missing from one file")
                )
            continue

        txt_smiles = txt_line.strip()
        csv_smiles = (csv_row["canonical_smiles"] or "").strip()

        if txt_smiles != csv_smiles:
            if len(alignment_errors) < 5:
                alignment_errors.append(
                    (row_num, txt_smiles[:80], csv_smiles[:80])
                )

        try:
            atoms = int(csv_row["num_atoms"])
            bonds = int(csv_row["num_bonds"])

            max_atoms = max(max_atoms, atoms)
            max_bonds = max(max_bonds, bonds)

            if bonds >= 119:
                large_molecules += 1

            fragment_counts[
                str(csv_row["fragmented"]).strip().lower()
            ] += 1

        except (ValueError, TypeError) as exc:
            if len(metadata_errors) < 5:
                metadata_errors.append((row_num, str(exc)))

print("\nDATASET SUMMARY")
print(f"Rows checked            : {total_rows:,}")
print(f"Missing records         : {missing_records:,}")
print(f"Alignment examples      : {alignment_errors}")
print(f"Metadata error examples : {metadata_errors}")

print("\nMOLECULAR SIZE")
print(f"Maximum atoms           : {max_atoms}")
print(f"Maximum bonds           : {max_bonds}")
print(f"Molecules with >=119 bonds: {large_molecules:,}")

print("\nFRAGMENT STATUS")
for key, value in fragment_counts.items():
    print(f"{key:12s}: {value:,}")

print("\nSTATUS:")
if missing_records or alignment_errors or metadata_errors:
    print("REVIEW REQUIRED — inconsistencies detected.")
else:
    print("PASS — no inconsistencies detected in the recorded checks.")


3.1 — DATASET CONSISTENCY AND SIZE AUDIT
CSV columns: ['chembl_id', 'original_smiles', 'canonical_smiles', 'num_atoms', 'num_bonds', 'fragmented']

DATASET SUMMARY
Rows checked            : 1,090,506
Missing records         : 0
Alignment examples      : []
Metadata error examples : []

MOLECULAR SIZE
Maximum atoms           : 48
Maximum bonds           : 49
Molecules with >=119 bonds: 0

FRAGMENT STATUS
false       : 1,072,135
true        : 18,371

STATUS:
PASS — no inconsistencies detected in the recorded checks.


In [23]:
from rdkit import Chem
from collections import Counter

print("=" * 65)
print("3.2 — RDKIT AND VOCABULARY COMPATIBILITY")
print("=" * 65)

SAMPLE_TARGET = 5000
stride = max(1, total_rows // SAMPLE_TARGET)

vocab_set = set(atom_types)

allowed_bonds = {
    Chem.BondType.AROMATIC,
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE,
}

stats = Counter()
unknown_atoms = Counter()
unknown_bonds = Counter()
examples = {
    "invalid": [],
    "fragmented": [],
    "oov": [],
    "metadata_mismatch": [],
}

def save_example(category, value, limit=5):
    if len(examples[category]) < limit:
        examples[category].append(value)

with CLEAN_TXT.open("r", encoding="utf-8-sig") as txt_file, \
     CLEAN_CSV.open("r", encoding="utf-8-sig", newline="") as csv_file:

    reader = csv.DictReader(csv_file)

    for i, (line, row) in enumerate(zip(txt_file, reader)):
        if i % stride != 0:
            continue

        smiles = line.strip()
        stats["sampled"] += 1

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            stats["invalid"] += 1
            save_example("invalid", smiles)
            continue

        stats["valid"] += 1

        if len(Chem.GetMolFrags(mol)) > 1:
            stats["fragmented"] += 1
            save_example("fragmented", smiles)

        if (
            mol.GetNumAtoms() != int(row["num_atoms"])
            or mol.GetNumBonds() != int(row["num_bonds"])
        ):
            stats["metadata_mismatch"] += 1
            save_example("metadata_mismatch", smiles)

        has_oov = False

        for atom in mol.GetAtoms():
            key = (
                atom.GetSymbol(),
                atom.GetFormalCharge(),
                atom.GetNumExplicitHs(),
            )

            if key not in vocab_set:
                unknown_atoms[key] += 1
                has_oov = True

        if has_oov:
            stats["molecules_with_oov"] += 1
            save_example("oov", smiles)

        has_unknown_bond = False

        for bond in mol.GetBonds():
            bond_type = bond.GetBondType()
            if bond_type not in allowed_bonds:
                unknown_bonds[str(bond_type)] += 1
                has_unknown_bond = True

        if has_unknown_bond:
            stats["molecules_with_unknown_bonds"] += 1

        if mol.GetNumBonds() >= 119:
            stats["large_graphs"] += 1

print("\nSAMPLE SUMMARY")
for key in [
    "sampled",
    "valid",
    "invalid",
    "fragmented",
    "metadata_mismatch",
    "molecules_with_oov",
    "molecules_with_unknown_bonds",
    "large_graphs",
]:
    print(f"{key:32s}: {stats[key]:,}")

if stats["sampled"]:
    validity = 100 * stats["valid"] / stats["sampled"]
    print(f"\nRDKit validity: {validity:.2f}%")

print("\nUNKNOWN ATOM TYPES")
print(unknown_atoms.most_common(15) or "None")

print("\nUNKNOWN BOND TYPES")
print(unknown_bonds.most_common() or "None")

print("\nISSUE EXAMPLES")
for category, records in examples.items():
    print(f"{category}: {records or 'None'}")

print("\nSTATUS:")
issues = [
    stats["invalid"],
    stats["fragmented"],
    stats["metadata_mismatch"],
    stats["molecules_with_oov"],
    stats["molecules_with_unknown_bonds"],
]

if any(issues):
    print("REVIEW REQUIRED — investigate before model construction.")
else:
    print("PASS — sampled molecules are compatible with graph encoding.")


3.2 — RDKIT AND VOCABULARY COMPATIBILITY

SAMPLE SUMMARY
sampled                         : 5,003
valid                           : 5,003
invalid                         : 0
fragmented                      : 0
metadata_mismatch               : 0
molecules_with_oov              : 0
molecules_with_unknown_bonds    : 0
large_graphs                    : 0

RDKit validity: 100.00%

UNKNOWN ATOM TYPES
None

UNKNOWN BOND TYPES
None

ISSUE EXAMPLES
invalid: None
fragmented: None
oov: None
metadata_mismatch: None

STATUS:
PASS — sampled molecules are compatible with graph encoding.


In [24]:
import csv
import random
from collections import Counter

print("=" * 65)
print("3.3 — FRAGMENTATION METADATA AUDIT")
print("=" * 65)

SAMPLE_PER_GROUP = 300
rng = random.Random(42)

fragment_samples = {True: [], False: []}
fragment_totals = Counter()
fragment_by_decile = Counter()
first_flagged = []
last_flagged = []

stride = max(1, total_rows // 5000)
previous_sample_flags = Counter()

with CLEAN_CSV.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    for index, record in enumerate(reader):
        flagged = record["fragmented"].strip().lower() == "true"

        fragment_totals[flagged] += 1

        decile = min(9, index * 10 // total_rows)
        fragment_by_decile[(decile, flagged)] += 1

        if index % stride == 0:
            previous_sample_flags[flagged] += 1

        if flagged:
            if len(first_flagged) < 3:
                first_flagged.append((index + 1, record))
            last_flagged.append((index + 1, record))
            last_flagged = last_flagged[-3:]

        # Reservoir sampling: random samples across the full file
        group = fragment_samples[flagged]
        seen = fragment_totals[flagged]

        if len(group) < SAMPLE_PER_GROUP:
            group.append(record)
        else:
            j = rng.randrange(seen)
            if j < SAMPLE_PER_GROUP:
                group[j] = record

print("\nFRAGMENT FLAG COUNTS")
for flag in [False, True]:
    count = fragment_totals[flag]
    print(f"fragmented={str(flag):5s}: {count:,} ({100*count/total_rows:.3f}%)")

print("\nDISTRIBUTION ACROSS DATASET")
for d in range(10):
    flagged_count = fragment_by_decile[(d, True)]
    print(f"Decile {d+1:2d}: {flagged_count:,} flagged records")

print("\nFLAGS IN PREVIOUS STRIDE SAMPLE")
print(dict(previous_sample_flags))

print("\nFIRST 3 FLAGGED EXAMPLES")
for row_number, record in first_flagged:
    print(f"\nRow {row_number}")
    print("Original :", record["original_smiles"][:160])
    print("Canonical:", record["canonical_smiles"][:160])

print("\nLAST FLAGGED ROW NUMBERS")
print([row_number for row_number, _ in last_flagged])

print("\nSTATUS: Fragmentation metadata inspected.")

3.3 — FRAGMENTATION METADATA AUDIT

FRAGMENT FLAG COUNTS
fragmented=False: 1,072,135 (98.315%)
fragmented=True : 18,371 (1.685%)

DISTRIBUTION ACROSS DATASET
Decile  1: 3,591 flagged records
Decile  2: 3,372 flagged records
Decile  3: 3,088 flagged records
Decile  4: 2,574 flagged records
Decile  5: 1,680 flagged records
Decile  6: 1,358 flagged records
Decile  7: 947 flagged records
Decile  8: 709 flagged records
Decile  9: 571 flagged records
Decile 10: 481 flagged records

FLAGS IN PREVIOUS STRIDE SAMPLE
{False: 4933, True: 70}

FIRST 3 FLAGGED EXAMPLES

Row 30
Original : Cc1ccc([N+](=O)[O-])c(OCCN2CCc3ccccc3C2)c1.O=C(O)C(=O)O
Canonical: Cc1ccc([N+](=O)[O-])c(OCCN2CCc3ccccc3C2)c1

Row 74
Original : CCc1c(O)c(=O)ccn1CCOC(=O)CCC(=O)CN.Cl
Canonical: CCc1c(O)c(=O)ccn1CCOC(=O)CCC(=O)CN

Row 81
Original : CCN(CC)CCNC(=O)c1ccc2cc([125I])ccc2n1.Cl.Cl
Canonical: CCN(CC)CCNC(=O)c1ccc2cc([125I])ccc2n1

LAST FLAGGED ROW NUMBERS
[1090367, 1090393, 1090466]

STATUS: Fragmentation metadata inspect

In [25]:
from rdkit import Chem
from collections import Counter

print("=" * 65)
print("3.4 — ORIGINAL VS CLEANED MOLECULES")
print("=" * 65)

fragment_analysis = {}

for flagged in [False, True]:
    records = fragment_samples[flagged]
    counts = Counter()
    examples = []

    for record in records:
        original = record["original_smiles"]
        cleaned = record["canonical_smiles"]

        mol_original = Chem.MolFromSmiles(original)
        mol_cleaned = Chem.MolFromSmiles(cleaned)

        counts["checked"] += 1

        if mol_original is None:
            counts["invalid_original"] += 1
            continue

        if mol_cleaned is None:
            counts["invalid_cleaned"] += 1
            continue

        original_frags = len(Chem.GetMolFrags(mol_original))
        cleaned_frags = len(Chem.GetMolFrags(mol_cleaned))

        if original_frags > 1:
            counts["original_multifragment"] += 1

        if cleaned_frags > 1:
            counts["cleaned_multifragment"] += 1

        if cleaned_frags < original_frags:
            counts["fragments_removed"] += 1

        if original != cleaned:
            counts["smiles_changed"] += 1

        if mol_cleaned.GetNumAtoms() < mol_original.GetNumAtoms():
            counts["atom_count_reduced"] += 1

        if (
            mol_cleaned.GetNumAtoms() != int(record["num_atoms"])
            or mol_cleaned.GetNumBonds() != int(record["num_bonds"])
        ):
            counts["metadata_mismatch"] += 1

        if original != cleaned and len(examples) < 5:
            examples.append({
                "original": original[:150],
                "cleaned": cleaned[:150],
                "original_fragments": original_frags,
                "cleaned_fragments": cleaned_frags,
            })

    fragment_analysis[flagged] = counts

    print(f"\n{'-' * 55}")
    print(f"GROUP: fragmented={flagged}")
    print(f"{'-' * 55}")

    for key in [
        "checked",
        "invalid_original",
        "invalid_cleaned",
        "original_multifragment",
        "cleaned_multifragment",
        "fragments_removed",
        "smiles_changed",
        "atom_count_reduced",
        "metadata_mismatch",
    ]:
        print(f"{key:30s}: {counts[key]}")

    print("\nExamples of changed SMILES:")
    for example in examples:
        print(example)

print("\nSTATUS: Molecular fragmentation comparison complete.")

3.4 — ORIGINAL VS CLEANED MOLECULES

-------------------------------------------------------
GROUP: fragmented=False
-------------------------------------------------------
checked                       : 300
invalid_original              : 0
invalid_cleaned               : 0
original_multifragment        : 0
cleaned_multifragment         : 0
fragments_removed             : 0
smiles_changed                : 2
atom_count_reduced            : 0
metadata_mismatch             : 0

Examples of changed SMILES:
{'original': 'CC(C)(C)N1C(=O)[C@@H]2[C@@H](CC[C@@H]3C(=O)C=C[C@@H](O)[C@H]32)C1=O', 'cleaned': 'CC(C)(C)N1C(=O)[C@H]2[C@@H]3[C@H](O)C=CC(=O)[C@H]3CC[C@H]2C1=O', 'original_fragments': 1, 'cleaned_fragments': 1}
{'original': 'C[C@@H]1CC[C@H]2C(COCC(O)CO)=C(C(F)(F)F)O[C@@H]3O[C@]4(C)CCC1[C@@]23OO4', 'cleaned': 'C[C@@H]1CC[C@H]2C(COCC(O)CO)=C(C(F)(F)F)O[C@@H]3O[C@]4(C)CCC1[C@]32OO4', 'original_fragments': 1, 'cleaned_fragments': 1}

-------------------------------------------------------
G

## Full Dataset Integrity Audit

In [26]:
from collections import Counter
import hashlib

print("=" * 65)
print("4.1 — DUPLICATE MOLECULE AND FILE INTEGRITY AUDIT")
print("=" * 65)

seen_smiles = set()
duplicate_count = 0
duplicate_examples = []
blank_count = 0
molecule_count = 0

with CLEAN_TXT.open("r", encoding="utf-8-sig") as f:
    for row_number, line in enumerate(f, 1):
        smiles = line.strip()

        if not smiles:
            blank_count += 1
            continue

        molecule_count += 1

        if smiles in seen_smiles:
            duplicate_count += 1
            if len(duplicate_examples) < 5:
                duplicate_examples.append((row_number, smiles))
        else:
            seen_smiles.add(smiles)

unique_count = len(seen_smiles)

def calculate_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

CHEMBL_TXT_HASH = calculate_sha256(CLEAN_TXT)
CHEMBL_CSV_HASH = calculate_sha256(CLEAN_CSV)

print("\nMOLECULE COUNTS")
print(f"Total nonempty records : {molecule_count:,}")
print(f"Unique SMILES          : {unique_count:,}")
print(f"Duplicate records      : {duplicate_count:,}")
print(f"Blank records          : {blank_count:,}")
print(f"Duplicate fraction (%) : {100 * duplicate_count / max(1, molecule_count):.4f}")

print("\nDUPLICATE EXAMPLES")
for row_number, smiles in duplicate_examples:
    print(f"Row {row_number}: {smiles}")
if not duplicate_examples:
    print("None")

print("\nDATASET FINGERPRINTS")
print(f"TXT SHA256: {CHEMBL_TXT_HASH}")
print(f"CSV SHA256: {CHEMBL_CSV_HASH}")
print(f"VOC SHA256: {ATOM_VOCAB_HASH}")

print("\nSTATUS")
if duplicate_count == 0 and blank_count == 0:
    print("PASS — no exact duplicate or blank SMILES.")
else:
    print("REVIEW REQUIRED — duplicates or blank records detected.")

del seen_smiles


4.1 — DUPLICATE MOLECULE AND FILE INTEGRITY AUDIT

MOLECULE COUNTS
Total nonempty records : 1,090,506
Unique SMILES          : 1,090,506
Duplicate records      : 0
Blank records          : 0
Duplicate fraction (%) : 0.0000

DUPLICATE EXAMPLES
None

DATASET FINGERPRINTS
TXT SHA256: a26351a9e8438e633c3824a21f3874609e71dc7d3a05c2f2055f7081c85d4b5b
CSV SHA256: f48d3601143dec710f55549d07f38f5b9f339e19f67c5af490dceb5503361660
VOC SHA256: 3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f

STATUS
PASS — no exact duplicate or blank SMILES.


In [28]:

from rdkit import Chem, RDLogger
from collections import Counter

print("=" * 65)
print("4.2 — FULL MOLECULAR GRAPH COMPATIBILITY AUDIT")
print("=" * 65)

vocab_set = set(atom_types)

allowed_bonds = {
    Chem.BondType.AROMATIC,
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE,
}

audit = Counter()
atom_frequency = Counter()
bond_frequency = Counter()
unknown_atom_frequency = Counter()
unknown_bond_frequency = Counter()

audit_examples = {
    "invalid": [],
    "disconnected": [],
    "oov": [],
    "unsupported_bond": [],
    "metadata_mismatch": [],
    "singleton": [],
}

def record_issue(category, row_number, smiles, limit=5):
    if len(audit_examples[category]) < limit:
        audit_examples[category].append(
            (row_number, smiles[:160])
        )

max_atoms_seen = 0
max_bonds_seen = 0
min_atoms_seen = float("inf")

RDLogger.DisableLog("rdApp.error")

try:
    with CLEAN_CSV.open(
        "r", encoding="utf-8-sig", newline=""
    ) as f:
        reader = csv.DictReader(f)

        for row_number, record in enumerate(reader, 1):
            smiles = record["canonical_smiles"].strip()
            audit["total"] += 1

            mol = Chem.MolFromSmiles(smiles)

            if mol is None:
                audit["invalid"] += 1
                record_issue("invalid", row_number, smiles)
                continue

            audit["valid"] += 1

            n_atoms = mol.GetNumAtoms()
            n_bonds = mol.GetNumBonds()

            max_atoms_seen = max(max_atoms_seen, n_atoms)
            max_bonds_seen = max(max_bonds_seen, n_bonds)
            min_atoms_seen = min(min_atoms_seen, n_atoms)

            if n_atoms == 1:
                audit["single_atom"] += 1
                record_issue("singleton", row_number, smiles)

            if len(Chem.GetMolFrags(mol)) != 1:
                audit["disconnected"] += 1
                record_issue("disconnected", row_number, smiles)

            if (
                n_atoms != int(record["num_atoms"])
                or n_bonds != int(record["num_bonds"])
            ):
                audit["metadata_mismatch"] += 1
                record_issue("metadata_mismatch", row_number, smiles)

            if n_bonds + 1 > 120:
                audit["exceeds_step_limit"] += 1

            molecule_has_oov = False
            for atom in mol.GetAtoms():
                key = (
                    atom.GetSymbol(),
                    atom.GetFormalCharge(),
                    atom.GetNumExplicitHs(),
                )
                atom_frequency[key] += 1

                if key not in vocab_set:
                    unknown_atom_frequency[key] += 1
                    molecule_has_oov = True

            if molecule_has_oov:
                audit["oov_molecules"] += 1
                record_issue("oov", row_number, smiles)

            molecule_has_bad_bond = False
            for bond in mol.GetBonds():
                bt = bond.GetBondType()
                bond_frequency[str(bt)] += 1

                if bt not in allowed_bonds:
                    unknown_bond_frequency[str(bt)] += 1
                    molecule_has_bad_bond = True

            if molecule_has_bad_bond:
                audit["unsupported_bond_molecules"] += 1
                record_issue("unsupported_bond", row_number, smiles)

            if audit["total"] % 200000 == 0:
                print(f"Scanned {audit['total']:,} molecules...", flush=True)

finally:
    RDLogger.EnableLog("rdApp.error")

print("\nFULL DATASET RESULTS")
for key in [
    "total",
    "valid",
    "invalid",
    "disconnected",
    "metadata_mismatch",
    "oov_molecules",
    "unsupported_bond_molecules",
    "single_atom",
    "exceeds_step_limit",
]:
    print(f"{key:32s}: {audit[key]:,}")

print("\nGRAPH SIZE")
print(f"Minimum atoms : {min_atoms_seen}")
print(f"Maximum atoms : {max_atoms_seen}")
print(f"Maximum bonds : {max_bonds_seen}")

print("\nVOCABULARY COVERAGE")
used_atom_types = set(atom_frequency)
unused_atom_types = vocab_set - used_atom_types

print(f"Vocabulary size       : {len(vocab_set)}")
print(f"Atom types observed   : {len(used_atom_types)}")
print(f"Unused vocabulary types: {len(unused_atom_types)}")
print(f"Unused entries        : {sorted(unused_atom_types)}")
print(f"Unknown atom types    : {unknown_atom_frequency.most_common(10)}")

print("\nBOND TYPE DISTRIBUTION")
for bond_type, count in bond_frequency.most_common():
    print(f"{bond_type:12s}: {count:,}")

print("\nISSUE EXAMPLES")
for category, examples_found in audit_examples.items():
    print(f"{category}: {examples_found or 'None'}")

print("\nSTATUS")
critical_issues = sum(audit[key] for key in [
    "invalid",
    "disconnected",
    "metadata_mismatch",
    "oov_molecules",
    "unsupported_bond_molecules",
    "exceeds_step_limit",
])

if critical_issues == 0:
    print("PASS — all scanned molecules satisfy core graph requirements.")
else:
    print("REVIEW REQUIRED — graph compatibility issues detected.")

if audit["single_atom"] > 0:
    print("NOTE — singleton graphs require explicit encoder testing.")


4.2 — FULL MOLECULAR GRAPH COMPATIBILITY AUDIT
Scanned 200,000 molecules...
Scanned 400,000 molecules...
Scanned 600,000 molecules...
Scanned 800,000 molecules...
Scanned 1,000,000 molecules...

FULL DATASET RESULTS
total                           : 1,090,506
valid                           : 1,090,506
invalid                         : 0
disconnected                    : 0
metadata_mismatch               : 0
oov_molecules                   : 0
unsupported_bond_molecules      : 0
single_atom                     : 0
exceeds_step_limit              : 0

GRAPH SIZE
Minimum atoms : 2
Maximum atoms : 48
Maximum bonds : 49

VOCABULARY COVERAGE
Vocabulary size       : 55
Atom types observed   : 35
Unused vocabulary types: 20
Unused entries        : [('B', -1, 0), ('B', -1, 3), ('B', 0, 0), ('Fe', -2, 0), ('Fe', 0, 0), ('Hg', 0, 0), ('K', 0, 0), ('Li', 0, 0), ('Na', 0, 0), ('O', 0, 1), ('O', 1, 1), ('Re', 0, 0), ('Ru', 0, 0), ('S', 1, 1), ('Sb', 0, 0), ('Se', 0, 0), ('Si', 0, 0), ('Sn', 0, 0), 

## BindingDB Leakage Verification

Find the authoritative BindingDB ligand files and determine how to compare them against our cleaned ChEMBL dataset.

The previous reproduction had a problem with BindingDB ligands leaking into pretraining. We need to independently verify that the new cleaned dataset does not have the same problem.

In [29]:
import os
import re
from pathlib import Path

print("=" * 65)
print("5.1 — BINDINGDB FILE DISCOVERY")
print("=" * 65)

SEARCH_ROOTS = [
    PROJECT_ROOT / "REWORKED_CODE" / "PREPROCESSING_OUTPUT",
    PROJECT_ROOT / "datasets" / "data" / "bindingdb",
]

SKIP_DIRS = {
    "needle_outputs", ".git", "__pycache__",
    ".ipynb_checkpoints", "node_modules",
    "pretraining_runs", "checkpoints",
}

pattern = re.compile(
    r"ligand|ligs|bindingdb|split_manifest|"
    r"exclusion|leakage|overlap",
    re.IGNORECASE
)

valid_suffixes = {
    "", ".txt", ".csv", ".tsv",
    ".json", ".jsonl", ".smi", ".smiles",
    ".parquet",
}

ligand_candidates = []

for root in SEARCH_ROOTS:
    print(f"\nSearch root: {root}")
    print(f"Exists     : {root.is_dir()}")

    if not root.is_dir():
        continue

    for current_dir, dirs, files in os.walk(root):
        depth = len(Path(current_dir).relative_to(root).parts)

        dirs[:] = [
            d for d in dirs
            if d not in SKIP_DIRS and depth < 6
        ]

        for filename in files:
            path = Path(current_dir) / filename

            if not pattern.search(filename):
                continue

            if path.suffix.lower() not in valid_suffixes:
                continue

            ligand_candidates.append(path)

ligand_candidates = sorted(set(ligand_candidates))

print("\nDISCOVERY RESULTS")
print(f"Candidate files found: {len(ligand_candidates)}")

for i, path in enumerate(ligand_candidates[:60], 1):
    try:
        size_mb = path.stat().st_size / 1024**2
        print(
            f"{i:3d}. {path.relative_to(PROJECT_ROOT)} "
            f"({size_mb:.2f} MB)"
        )
    except OSError as exc:
        print(f"{i:3d}. {path} — {exc}")

if len(ligand_candidates) > 60:
    print(f"\n... {len(ligand_candidates)-60} additional candidates")

print("\nSTATUS: File discovery completed.")

5.1 — BINDINGDB FILE DISCOVERY

Search root: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT
Exists     : True

Search root: /home/muhamed/repo/CTDDG/datasets/data/bindingdb
Exists     : True

DISCOVERY RESULTS
Candidate files found: 14
  1. REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/misc/chembl_no_bindingdb_overlap.csv (125.10 MB)
  2. REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/misc/chembl_standardized_pre_overlap.csv (133.67 MB)
  3. datasets/data/bindingdb/ligand_ames_labels.csv (6.81 MB)
  4. datasets/data/bindingdb/ligand_ames_labels.csv.meta.json (0.00 MB)
  5. datasets/data/bindingdb/test_dataset/test_4_org_1000_112/ligands_test_4_org_corrected (1.82 MB)
  6. datasets/data/bindingdb/test_dataset/test_4_org_1002_103/ligands_test_4_org_corrected (1.82 MB)
  7. datasets/data/bindingdb/test_dataset/test_4_org_1004_122/ligands_test_4_org_corrected (2.92 MB)
  8. datasets/data/bindingdb/test_dataset/test_4_org_1036_124/ligands_test_4_org_corrected (2.87 MB)
  9. datasets/dat

In [30]:
print("=" * 65)
print("5.2 — BINDINGDB LIGAND FORMAT INSPECTION")
print("=" * 65)

ligand_files = [
    p for p in ligand_candidates
    if re.search(r"ligand|ligs", p.name, re.IGNORECASE)
]

print(f"Candidate ligand files: {len(ligand_files)}")

if not ligand_files:
    print("\nNo ligand files detected in the searched locations.")

for i, path in enumerate(ligand_files[:12], 1):
    print(f"\n{'-' * 55}")
    print(f"FILE {i}: {path.relative_to(PROJECT_ROOT)}")
    print(f"{'-' * 55}")

    try:
        with path.open(
            "r", encoding="utf-8-sig", errors="replace"
        ) as f:
            for line_number in range(1, 4):
                line = f.readline()
                if not line:
                    break
                print(
                    f"Line {line_number}: "
                    f"{line.rstrip()[:180]!r}"
                )
    except Exception as exc:
        print(f"Preview unavailable: {exc}")

if len(ligand_files) > 12:
    print(f"\nAdditional ligand files: {len(ligand_files)-12}")

print("\nSTATUS: Format inspection completed.")

5.2 — BINDINGDB LIGAND FORMAT INSPECTION
Candidate ligand files: 12

-------------------------------------------------------
FILE 1: datasets/data/bindingdb/ligand_ames_labels.csv
-------------------------------------------------------
Line 1: 'smiles,ames'
Line 2: 'B[P@@](=O)(OC[C@H]1O[C@@H](n2cc(OC)c(=O)[nH]c2=O)[C@H](O)[C@@H]1O)OP(=O)(O)O,0'
Line 3: 'B[P@@](=O)(OC[C@H]1O[C@@H](n2cc(OC)c(=O)[nH]c2=O)[C@H](O)[C@@H]1O)OP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n2cc(OC)c(=O)[nH]c2=O)[C@H](O)[C@@H]1O,0'

-------------------------------------------------------
FILE 2: datasets/data/bindingdb/ligand_ames_labels.csv.meta.json
-------------------------------------------------------
Line 1: '{'
Line 2: ' "v": 3,'
Line 3: ' "rf": {'

-------------------------------------------------------
FILE 3: datasets/data/bindingdb/test_dataset/test_4_org_1000_112/ligands_test_4_org_corrected
-------------------------------------------------------
Line 1: 'C / C ( = N \\ O C C O ) c 1 c c c 2 n n c ( [ C @ @ H ] (

In [31]:

from rdkit import Chem, RDLogger
from collections import Counter
from pathlib import Path

print("=" * 65)
print("5.3 — BINDINGDB LIGAND RECONSTRUCTION")
print("=" * 65)

BINDINGDB_ROOT = PROJECT_ROOT / "datasets" / "data" / "bindingdb"

train_files = sorted(
    (BINDINGDB_ROOT / "train_dataset").glob(
        "train_*/ligands_train_4_org_corrected"
    )
)

test_files = sorted(
    (BINDINGDB_ROOT / "test_dataset").glob(
        "test_*/ligands_test_4_org_corrected"
    )
)

BINDINGDB_FILES = train_files + test_files

assert len(train_files) == 5, "Expected 5 train ligand files"
assert len(test_files) == 5, "Expected 5 test ligand files"

bindingdb_canonical = set()
bindingdb_parent_canonical = set()
bindingdb_by_file = {}

invalid_examples = []
stats = Counter()

# Cache repeated molecules across folds.
canonical_cache = {}

RDLogger.DisableLog("rdApp.error")

try:
    for path in BINDINGDB_FILES:
        file_stats = Counter()
        file_molecules = set()

        with path.open(
            "r", encoding="utf-8-sig"
        ) as f:
            for row_number, raw_line in enumerate(f, 1):
                file_stats["total"] += 1

                smiles = "".join(raw_line.split())

                if not smiles:
                    file_stats["blank"] += 1
                    continue

                if smiles not in canonical_cache:
                    mol = Chem.MolFromSmiles(smiles)

                    if mol is None:
                        canonical_cache[smiles] = None
                    else:
                        canonical = Chem.MolToSmiles(
                            mol, canonical=True, isomericSmiles=True
                        )

                        fragments = Chem.GetMolFrags(
                            mol, asMols=True
                        )
                        parent = max(
                            fragments,
                            key=lambda m: (
                                m.GetNumHeavyAtoms(),
                                m.GetNumAtoms()
                            )
                        )
                        parent_smiles = Chem.MolToSmiles(
                            parent, canonical=True, isomericSmiles=True
                        )

                        canonical_cache[smiles] = (
                            canonical,
                            parent_smiles,
                            len(fragments)
                        )

                result = canonical_cache[smiles]

                if result is None:
                    file_stats["invalid"] += 1
                    if len(invalid_examples) < 10:
                        invalid_examples.append(
                            (str(path), row_number, smiles[:160])
                        )
                    continue

                canonical, parent_smiles, fragment_count = result

                file_stats["valid"] += 1
                if fragment_count > 1:
                    file_stats["multifragment"] += 1

                file_molecules.add(canonical)
                bindingdb_canonical.add(canonical)
                bindingdb_parent_canonical.add(parent_smiles)

        file_stats["unique"] = len(file_molecules)
        stats.update({
            "total": stats["total"] + file_stats["total"],
            "valid": stats["valid"] + file_stats["valid"],
            "invalid": stats["invalid"] + file_stats["invalid"],
            "multifragment": (
                stats["multifragment"] + file_stats["multifragment"]
            ),
        })

        bindingdb_by_file[str(path.relative_to(PROJECT_ROOT))] = file_molecules

        print(f"\n{path.parent.name}")
        print(f"  Total records   : {file_stats['total']:,}")
        print(f"  Valid records   : {file_stats['valid']:,}")
        print(f"  Invalid records : {file_stats['invalid']:,}")
        print(f"  Unique molecules: {file_stats['unique']:,}")
        print(f"  Multifragment   : {file_stats['multifragment']:,}")

finally:
    RDLogger.EnableLog("rdApp.error")

print("\n" + "=" * 65)
print("OVERALL BINDINGDB SUMMARY")
print("=" * 65)

print(f"Total ligand records  : {stats['total']:,}")
print(f"Valid records         : {stats['valid']:,}")
print(f"Invalid records       : {stats['invalid']:,}")
print(f"Multifragment records : {stats['multifragment']:,}")
print(f"Unique canonical      : {len(bindingdb_canonical):,}")
print(f"Unique parent forms   : {len(bindingdb_parent_canonical):,}")
print(f"Invalid examples      : {invalid_examples}")

if stats["invalid"] == 0:
    print("\nSTATUS: PASS — all BindingDB ligands parsed successfully.")
else:
    print("\nSTATUS: REVIEW REQUIRED — invalid ligands found.")


5.3 — BINDINGDB LIGAND RECONSTRUCTION

train_4_org_1000_112
  Total records   : 143,999
  Valid records   : 143,999
  Invalid records : 0
  Unique molecules: 98,644
  Multifragment   : 570

train_4_org_1002_103
  Total records   : 144,885
  Valid records   : 144,885
  Invalid records : 0
  Unique molecules: 99,398
  Multifragment   : 558

train_4_org_1004_122
  Total records   : 146,059
  Valid records   : 146,059
  Invalid records : 0
  Unique molecules: 100,259
  Multifragment   : 571

train_4_org_1036_124
  Total records   : 142,687
  Valid records   : 142,687
  Invalid records : 0
  Unique molecules: 97,334
  Multifragment   : 566

train_4_org_1042_104
  Total records   : 227,093
  Valid records   : 227,093
  Invalid records : 0
  Unique molecules: 147,696
  Multifragment   : 582

test_4_org_1000_112
  Total records   : 16,200
  Valid records   : 16,200
  Invalid records : 0
  Unique molecules: 9,448
  Multifragment   : 18

test_4_org_1002_103
  Total records   : 16,015
  Valid rec

In [32]:
from collections import Counter

print("=" * 65)
print("5.4 — CHEMBL / BINDINGDB LEAKAGE VERIFICATION")
print("=" * 65)

assert stats["invalid"] == 0, (
    "Resolve invalid BindingDB SMILES before running overlap audit."
)

overlap = Counter()
overlap_by_file = Counter()
overlap_examples = []

bindingdb_raw_reconstructed = set(canonical_cache.keys())

with CLEAN_TXT.open("r", encoding="utf-8-sig") as f:
    for row_number, line in enumerate(f, 1):
        smiles = line.strip()
        overlap["chembl_total"] += 1

        if smiles in bindingdb_raw_reconstructed:
            overlap["direct"] += 1

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            overlap["chembl_invalid"] += 1
            continue

        canonical = Chem.MolToSmiles(
            mol, canonical=True, isomericSmiles=True
        )

        if canonical in bindingdb_canonical:
            overlap["canonical"] += 1

            if len(overlap_examples) < 10:
                overlap_examples.append(
                    (row_number, canonical)
                )

        if canonical in bindingdb_parent_canonical:
            overlap["parent_fragment"] += 1

        for filename, molecule_set in bindingdb_by_file.items():
            if canonical in molecule_set:
                overlap_by_file[filename] += 1

        if row_number % 200000 == 0:
            print(f"Compared {row_number:,} ChEMBL molecules...", flush=True)

print("\nOVERLAP SUMMARY")
print(f"ChEMBL molecules       : {overlap['chembl_total']:,}")
print(f"Direct SMILES matches  : {overlap['direct']:,}")
print(f"Canonical matches      : {overlap['canonical']:,}")
print(f"Parent-fragment matches: {overlap['parent_fragment']:,}")
print(f"Invalid ChEMBL records : {overlap['chembl_invalid']:,}")

print("\nOVERLAP BY BINDINGDB FILE")
for filename in sorted(bindingdb_by_file):
    print(f"{filename}: {overlap_by_file[filename]:,}")

print("\nCANONICAL OVERLAP EXAMPLES")
for row_number, smiles in overlap_examples:
    print(f"Row {row_number}: {smiles}")

print("\nSTATUS")

if (
    overlap["chembl_invalid"] == 0
    and overlap["canonical"] == 0
    and overlap["parent_fragment"] == 0
):
    print("PASS — no exact, canonical, or parent-fragment overlap detected.")
else:
    print("REVIEW REQUIRED — investigate detected overlaps.")


5.4 — CHEMBL / BINDINGDB LEAKAGE VERIFICATION
Compared 200,000 ChEMBL molecules...
Compared 400,000 ChEMBL molecules...
Compared 600,000 ChEMBL molecules...
Compared 800,000 ChEMBL molecules...
Compared 1,000,000 ChEMBL molecules...

OVERLAP SUMMARY
ChEMBL molecules       : 1,090,506
Direct SMILES matches  : 0
Canonical matches      : 0
Parent-fragment matches: 0
Invalid ChEMBL records : 0

OVERLAP BY BINDINGDB FILE
datasets/data/bindingdb/test_dataset/test_4_org_1000_112/ligands_test_4_org_corrected: 0
datasets/data/bindingdb/test_dataset/test_4_org_1002_103/ligands_test_4_org_corrected: 0
datasets/data/bindingdb/test_dataset/test_4_org_1004_122/ligands_test_4_org_corrected: 0
datasets/data/bindingdb/test_dataset/test_4_org_1036_124/ligands_test_4_org_corrected: 0
datasets/data/bindingdb/test_dataset/test_4_org_1042_104/ligands_test_4_org_corrected: 0
datasets/data/bindingdb/train_dataset/train_4_org_1000_112/ligands_train_4_org_corrected: 0
datasets/data/bindingdb/train_dataset/train

## Reproducible Train/Validation Split

In [33]:
import hashlib
import numpy as np

print("=" * 65)
print("6.1 — PRETRAINING SPLIT CONFIGURATION")
print("=" * 65)

# Dataset configuration
PRETRAIN_DATA_PATH = CLEAN_TXT
PRETRAIN_VOCAB_PATH = ATOM_TYPES_PATH

EXPECTED_MOLECULES = 1_090_506
EXPECTED_DATA_SHA256 = (
    "a26351a9e8438e633c3824a21f3874609e71dc7d3a05c2f2055f7081c85d4b5b"
)
EXPECTED_VOCAB_SHA256 = (
    "3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f"
)

# Split configuration
SPLIT_SEED = 42
VALIDATION_FRACTION = 0.05
TRAIN_FRACTION = 1.0 - VALIDATION_FRACTION

# Training objective
OBJECTIVE = "unconditional_nll"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

data_hash = sha256_file(PRETRAIN_DATA_PATH)
vocab_hash = sha256_file(PRETRAIN_VOCAB_PATH)

assert data_hash == EXPECTED_DATA_SHA256, (
    "ChEMBL dataset changed since the full audit."
)
assert vocab_hash == EXPECTED_VOCAB_SHA256, (
    "Atom vocabulary changed since the full audit."
)
assert len(atom_types) == 55, "Unexpected vocabulary size."

print("\nINPUT VERIFICATION")
print(f"Dataset          : {PRETRAIN_DATA_PATH.name}")
print(f"Expected records : {EXPECTED_MOLECULES:,}")
print(f"Atom types       : {len(atom_types)}")
print(f"Dataset SHA256   : {data_hash}")
print(f"Vocabulary SHA256: {vocab_hash}")

print("\nSPLIT CONFIGURATION")
print(f"Training fraction  : {TRAIN_FRACTION:.1%}")
print(f"Validation fraction: {VALIDATION_FRACTION:.1%}")
print(f"Random seed        : {SPLIT_SEED}")
print(f"Objective          : {OBJECTIVE}")

print("\nSTATUS: PASS — configuration and fingerprints verified.")

6.1 — PRETRAINING SPLIT CONFIGURATION

INPUT VERIFICATION
Dataset          : chembl_clean.txt
Expected records : 1,090,506
Atom types       : 55
Dataset SHA256   : a26351a9e8438e633c3824a21f3874609e71dc7d3a05c2f2055f7081c85d4b5b
Vocabulary SHA256: 3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f

SPLIT CONFIGURATION
Training fraction  : 95.0%
Validation fraction: 5.0%
Random seed        : 42
Objective          : unconditional_nll

STATUS: PASS — configuration and fingerprints verified.


In [34]:
print("=" * 65)
print("6.2 — TRAIN / VALIDATION SPLIT")
print("=" * 65)

N_MOLECULES = EXPECTED_MOLECULES

rng = np.random.default_rng(SPLIT_SEED)

n_val = round(N_MOLECULES * VALIDATION_FRACTION)

# Select held-out molecule indices.
val_indices = np.sort(
    rng.choice(
        N_MOLECULES,
        size=n_val,
        replace=False
    )
).astype(np.int32)

# Remaining molecules become training data.
train_mask = np.ones(N_MOLECULES, dtype=bool)
train_mask[val_indices] = False

train_indices = np.flatnonzero(train_mask).astype(np.int32)

# Verify partition properties.
assert len(train_indices) + len(val_indices) == N_MOLECULES
assert len(np.unique(val_indices)) == len(val_indices)
assert np.all(~train_mask[val_indices])
assert np.all(train_mask[train_indices])
assert len(train_indices) == int(train_mask.sum())

# Fingerprint the selected validation indices.
SPLIT_HASH = hashlib.sha256(
    val_indices.tobytes()
).hexdigest()

print("\nSPLIT RESULTS")
print(f"Total molecules  : {N_MOLECULES:,}")
print(f"Training set     : {len(train_indices):,}")
print(f"Validation set   : {len(val_indices):,}")

print(f"\nTraining ratio   : {len(train_indices)/N_MOLECULES:.4%}")
print(f"Validation ratio : {len(val_indices)/N_MOLECULES:.4%}")

print("\nINTEGRITY CHECKS")
print(f"All records assigned       : PASS")
print(f"No shared row indices      : PASS")
print(f"No duplicate val indices   : PASS")
print(f"Validation index SHA256    : {SPLIT_HASH}")

print("\nFIRST 10 VALIDATION INDICES")
print(val_indices[:10].tolist())

print("\nSTATUS: PASS — deterministic split created.")

6.2 — TRAIN / VALIDATION SPLIT

SPLIT RESULTS
Total molecules  : 1,090,506
Training set     : 1,035,981
Validation set   : 54,525

Training ratio   : 95.0000%
Validation ratio : 5.0000%

INTEGRITY CHECKS
All records assigned       : PASS
No shared row indices      : PASS
No duplicate val indices   : PASS
Validation index SHA256    : c0126bcc5ba1bc930ef6a947ee87f73c8cb61c01940a65eb60b96179ab517526

FIRST 10 VALIDATION INDICES
[0, 3, 14, 20, 45, 48, 53, 114, 133, 144]

STATUS: PASS — deterministic split created.


In [35]:
import csv

print("=" * 65)
print("6.3 — TRAIN / VALIDATION DISTRIBUTION AUDIT")
print("=" * 65)

atom_counts = np.empty(N_MOLECULES, dtype=np.uint8)
bond_counts = np.empty(N_MOLECULES, dtype=np.uint8)
fragment_flags = np.empty(N_MOLECULES, dtype=bool)

with CLEAN_CSV.open(
    "r", encoding="utf-8-sig", newline=""
) as f:
    reader = csv.DictReader(f)

    rows_read = 0

    for i, record in enumerate(reader):
        if i >= N_MOLECULES:
            raise ValueError("CSV has more records than expected.")

        atom_counts[i] = int(record["num_atoms"])
        bond_counts[i] = int(record["num_bonds"])
        fragment_flags[i] = (
            record["fragmented"].strip().lower() == "true"
        )

        rows_read += 1

assert rows_read == N_MOLECULES, (
    f"Expected {N_MOLECULES:,}, found {rows_read:,}"
)

def describe_split(name, indices):
    atoms = atom_counts[indices]
    bonds = bond_counts[indices]
    fragments = fragment_flags[indices]

    result = {
        "count": len(indices),
        "mean_atoms": float(atoms.mean()),
        "median_atoms": float(np.median(atoms)),
        "p95_atoms": float(np.percentile(atoms, 95)),
        "max_atoms": int(atoms.max()),
        "mean_bonds": float(bonds.mean()),
        "median_bonds": float(np.median(bonds)),
        "p95_bonds": float(np.percentile(bonds, 95)),
        "max_bonds": int(bonds.max()),
        "fragmented_pct": float(fragments.mean() * 100),
    }

    print(f"\n{name.upper()}")
    print("-" * 45)

    for key, value in result.items():
        if key == "count":
            print(f"{key:22s}: {value:,}")
        else:
            print(f"{key:22s}: {value:.3f}")

    return result

train_distribution = describe_split("Training", train_indices)
val_distribution = describe_split("Validation", val_indices)

print("\nDISTRIBUTION DIFFERENCES")
print("-" * 45)

atom_diff = abs(
    train_distribution["mean_atoms"]
    - val_distribution["mean_atoms"]
)

bond_diff = abs(
    train_distribution["mean_bonds"]
    - val_distribution["mean_bonds"]
)

fragment_diff = abs(
    train_distribution["fragmented_pct"]
    - val_distribution["fragmented_pct"]
)

print(f"Mean atom count difference : {atom_diff:.4f}")
print(f"Mean bond count difference : {bond_diff:.4f}")
print(f"Fragmentation difference   : {fragment_diff:.4f} percentage points")

print("\nSTATUS")
if atom_diff < 1.0 and bond_diff < 1.0 and fragment_diff < 1.0:
    print("PASS — training and validation distributions are comparable.")
else:
    print("REVIEW — distribution differences need investigation.")

print("\nNo files have been written.")

6.3 — TRAIN / VALIDATION DISTRIBUTION AUDIT

TRAINING
---------------------------------------------
count                 : 1,035,981
mean_atoms            : 24.341
median_atoms          : 24.000
p95_atoms             : 33.000
max_atoms             : 48.000
mean_bonds            : 26.464
median_bonds          : 27.000
p95_bonds             : 36.000
max_bonds             : 49.000
fragmented_pct        : 1.681

VALIDATION
---------------------------------------------
count                 : 54,525
mean_atoms            : 24.301
median_atoms          : 24.000
p95_atoms             : 33.000
max_atoms             : 38.000
mean_bonds            : 26.420
median_bonds          : 27.000
p95_bonds             : 36.000
max_bonds             : 45.000
fragmented_pct        : 1.761

DISTRIBUTION DIFFERENCES
---------------------------------------------
Mean atom count difference : 0.0398
Mean bond count difference : 0.0439
Fragmentation difference   : 0.0800 percentage points

STATUS
PASS — training

In [36]:
import json
import os
import tempfile

print("=" * 65)
print("6.4 — SAVE REPRODUCIBLE DATASET SPLIT")
print("=" * 65)

SPLIT_DIR = (
    PROJECT_ROOT
    / "REWORKED_CODE"
    / "PRETRAINING_OUTPUT"
    / "no_toxicity"
    / "splits"
)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

split_name = (
    f"chembl_clean_seed{SPLIT_SEED}"
    f"_val5_{SPLIT_HASH[:12]}"
)

SPLIT_NPZ_PATH = SPLIT_DIR / f"{split_name}.npz"
SPLIT_MANIFEST_PATH = SPLIT_DIR / f"{split_name}.json"

manifest = {
    "schema_version": 1,
    "objective": "cdgcn_nll",
    "dataset_path": str(PRETRAIN_DATA_PATH.resolve()),
    "dataset_sha256": data_hash,
    "vocabulary_path": str(PRETRAIN_VOCAB_PATH.resolve()),
    "vocabulary_sha256": vocab_hash,
    "total_molecules": N_MOLECULES,
    "train_molecules": len(train_indices),
    "validation_molecules": len(val_indices),
    "split_seed": SPLIT_SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "split_method": "random_molecule_holdout",
    "validation_indices_sha256": SPLIT_HASH,
}

assert len(train_indices) + len(val_indices) == N_MOLECULES

if SPLIT_NPZ_PATH.exists() or SPLIT_MANIFEST_PATH.exists():
    assert SPLIT_NPZ_PATH.is_file()
    assert SPLIT_MANIFEST_PATH.is_file()

    with SPLIT_MANIFEST_PATH.open("r") as f:
        existing_manifest = json.load(f)

    assert existing_manifest == manifest, "Existing manifest differs."

    with np.load(SPLIT_NPZ_PATH, allow_pickle=False) as existing:
        assert np.array_equal(existing["train_indices"], train_indices)
        assert np.array_equal(existing["val_indices"], val_indices)

    print("Existing split verified; no files overwritten.")

else:
    # Write temporary files first, then publish the completed artifacts.
    temp_npz = None
    temp_json = None

    try:
        with tempfile.NamedTemporaryFile(
            dir=SPLIT_DIR, suffix=".npz", delete=False
        ) as f:
            temp_npz = Path(f.name)
            np.savez_compressed(
                f,
                train_indices=train_indices,
                val_indices=val_indices,
            )

        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8",
            dir=SPLIT_DIR, suffix=".json", delete=False
        ) as f:
            temp_json = Path(f.name)
            json.dump(manifest, f, indent=2, sort_keys=True)

        os.replace(temp_npz, SPLIT_NPZ_PATH)
        os.replace(temp_json, SPLIT_MANIFEST_PATH)

        print("New split artifacts created.")

    finally:
        for temporary in [temp_npz, temp_json]:
            if temporary is not None:
                temporary.unlink(missing_ok=True)

print("\nSAVED ARTIFACTS")
print(f"Indices  : {SPLIT_NPZ_PATH}")
print(f"Manifest : {SPLIT_MANIFEST_PATH}")
print(f"Train    : {len(train_indices):,}")
print(f"Val      : {len(val_indices):,}")
print(f"Hash     : {SPLIT_HASH}")

print("\nSTATUS: Split persistence completed.")

6.4 — SAVE REPRODUCIBLE DATASET SPLIT
New split artifacts created.

SAVED ARTIFACTS
Indices  : /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/splits/chembl_clean_seed42_val5_c0126bcc5ba1.npz
Manifest : /home/muhamed/repo/CTDDG/REWORKED_CODE/PRETRAINING_OUTPUT/no_toxicity/splits/chembl_clean_seed42_val5_c0126bcc5ba1.json
Train    : 1,035,981
Val      : 54,525
Hash     : c0126bcc5ba1bc930ef6a947ee87f73c8cb61c01940a65eb60b96179ab517526

STATUS: Split persistence completed.


In [37]:
print("=" * 65)
print("6.5 — SAVED SPLIT REPRODUCIBILITY TEST")
print("=" * 65)

with SPLIT_MANIFEST_PATH.open("r", encoding="utf-8") as f:
    saved_manifest = json.load(f)

with np.load(SPLIT_NPZ_PATH, allow_pickle=False) as saved:
    loaded_train = saved["train_indices"]
    loaded_val = saved["val_indices"]

checks = {
    "Dataset hash matches":
        saved_manifest["dataset_sha256"] == data_hash,

    "Vocabulary hash matches":
        saved_manifest["vocabulary_sha256"] == vocab_hash,

    "Training indices identical":
        np.array_equal(loaded_train, train_indices),

    "Validation indices identical":
        np.array_equal(loaded_val, val_indices),

    "All records accounted for":
        len(loaded_train) + len(loaded_val) == N_MOLECULES,

    "No shared indices":
        len(np.intersect1d(loaded_train, loaded_val)) == 0,

    "All indices in range":
        (
            loaded_train.min() >= 0
            and loaded_val.min() >= 0
            and loaded_train.max() < N_MOLECULES
            and loaded_val.max() < N_MOLECULES
        ),

    "Validation hash matches":
        hashlib.sha256(
            loaded_val.tobytes()
        ).hexdigest() == saved_manifest["validation_indices_sha256"],
}

print("\nREPRODUCIBILITY CHECKS")
for name, passed in checks.items():
    print(f"{name:35s}: {'PASS' if passed else 'FAIL'}")

print("\nSUMMARY")
print(f"Train records     : {len(loaded_train):,}")
print(f"Validation records: {len(loaded_val):,}")
print(f"All checks passed : {sum(checks.values())}/{len(checks)}")

assert all(checks.values()), "Saved split verification failed."

print("\nSTATUS: PASS — saved split is reproducible.")

6.5 — SAVED SPLIT REPRODUCIBILITY TEST

REPRODUCIBILITY CHECKS
Dataset hash matches               : PASS
Vocabulary hash matches            : PASS
Training indices identical         : PASS
Validation indices identical       : PASS
All records accounted for          : PASS
No shared indices                  : PASS
All indices in range               : PASS
Validation hash matches            : PASS

SUMMARY
Train records     : 1,035,981
Validation records: 54,525
All checks passed : 8/8

STATUS: PASS — saved split is reproducible.
